In [1]:
import pyroomacoustics as pra

import os
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.autograd import profiler
import torchaudio
from torchmetrics.audio import SpeechReverberationModulationEnergyRatio, ShortTimeObjectiveIntelligibility, DeepNoiseSuppressionMeanOpinionScore


from einops import rearrange

from src.dataset import SignalDataset, TRUNetDataset
from src.loss import loss_tot, loss_MR, loss_MR_w
from models.fspen import * # FullSubPathExtension, FullSubPathExtension_3_heads, FullSubPathExtension_ver2, FullSubPathExtension_abs_pha, FullSubPathExtension_abs_pha_mapping, FullSubPathExtension_ver2_abs_pha, FullSubPathExtension_ver3

from IPython.display import Audio

from src.utils import model_eval, model_eval_fspen2x_ver3, model_eval_3_heads, use_pcs, inv_pcs, model_eval_old

import matplotlib.pyplot as plt

In [2]:
TEST_DIR = os.path.join("data", "DS_10283_2791", "clean_testset_wav")
TEST_NOISE_DIR = os.path.join("data", "DS_10283_2791", "noisy_testset_wav")
NOISE_DIR = os.path.join("data", "demand_test")

CHKP_DIR = "checkpoints"

np.set_printoptions(precision=3)
torch.set_printoptions(precision=3)

In [3]:
SEED = 1984

np.random.seed(SEED)
torch.manual_seed(SEED)

gen = torch.Generator()
gen.manual_seed(SEED)

In [4]:
# N_FFTS = 512
# HOP_LENGTH = 256
# HID_SIZE = 32
# SR = 16_000

N_FFTS = 512 # configs.n_fft
HOP_LENGTH = 256 # configs.hop_length
SR = 16_000
BATCH_SIZE = 8 # 32

DEVICE = "cpu" # torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"It's {DEVICE} time!!!")

It's cpu time!!!


In [5]:
from gtcrn.gtcrn import GTCRN

model = GTCRN()
ckpt = torch.load(os.path.join('gtcrn', 'checkpoints', 'model_trained_on_dns3.tar'), map_location=DEVICE)
model.load_state_dict(ckpt['model'])

<All keys matched successfully>

In [6]:
rir_dict = {1: os.path.join("data", "rirs48_small_3_test"), 1: os.path.join("data", "rirs48_medium_3_test"), 1: os.path.join("data", "rirs48_large_3_test"), 1: os.path.join("data", "rirs48_super_large_3_test")}
dataset = TRUNetDataset(TEST_DIR, sr=SR, noise_dir=NOISE_DIR, rir_dir=rir_dict, snr=[0, 5, 10, 15], rir_proba=0.85, noise_proba=0.85, rir_target=False, return_noise=False, return_rir=False, shuffle_files=False)
dataset.set_epoch(99)

36
12


In [7]:
def vorbis_window(winlen, device="cuda"):
    sq = torch.sin(torch.pi/2*(torch.sin(torch.pi/winlen*(torch.arange(winlen)-0.5))**2)).float()
    return sq

In [8]:
import yaml

from NISQA_s.src.core.model_torch import model_init
from NISQA_s.src.utils.process_utils import process

NISQA_PATH = "NISQA_s/config/nisqa_s.yaml"

with open(NISQA_PATH, 'r') as stream:
    nisqa_args = yaml.safe_load(stream)
nisqa_args["ms_n_fft"] = 512
nisqa_args["hop_length"] = 256
nisqa_args["ms_win_length"] = 512
nisqa_args["ckp"] = nisqa_args["ckp"][3:]

nisqa, h0_nisqa, c0_nisqa = model_init(nisqa_args)

/home/zakhar/miniconda3/envs/ems_dereverb/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=1 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [9]:
def pad_sequence(batch):
    if not batch:
        return torch.zeros(0), torch.zeros(0)

    input_signal, target_signal, noise, rir = zip(*batch)
        
    max_len_s = max(s.shape[-1] for s in input_signal)
    
    padded_input = torch.zeros(len(input_signal), max_len_s)
    padded_target = torch.zeros(len(target_signal), max_len_s)
    
    for i, s in enumerate(input_signal):
        padded_input[i, :s.shape[-1]] = s
        padded_target[i, :s.shape[-1]] = target_signal[i]

    return padded_input, padded_target


def collate_fn(batch):
    
    padded_input, padded_target = pad_sequence(batch)
        
    padded_input = padded_input.reshape(-1, padded_input.shape[-1])
    padded_target = padded_target.reshape(-1, padded_input.shape[-1])

    return padded_input, padded_target

In [10]:
test_dataloader = DataLoader(dataset, batch_size=1, shuffle=False, drop_last=False, collate_fn=collate_fn)

In [11]:
import time

def check_stream_inference(model, loader, window_size = 1 * 48_000 // 4, device="cpu"):
    model.eval()

    result_nisqa_full = []
    result_rtf_full = []
    result_nisqa_chunk = []
    result_rtf_chunk = []
    with torch.no_grad():
        for signal, target in tqdm(loader):
            signal = signal.to(device)
            target = target.to(device)
            window = vorbis_window(N_FFTS).to(device)
    
            start_time = time.time()
            spec = torch.stft(
                signal,
                n_fft=N_FFTS,
                hop_length=HOP_LENGTH,
                # onesided=True,
                win_length=N_FFTS,
                window=window,
                return_complex=True,
                normalized=True,
                center=True
            ) 

            # spec = use_pcs(spec, N_FFTS)
            
            input_spec = torch.view_as_real(spec)

            output = model(input_spec)

            output = torch.view_as_complex(output.contiguous()) # model_eval(model, spec, configs, device, hid_size=HID_SIZE)

            # output = inv_pcs(output.abs(), output.angle())

            window = vorbis_window(N_FFTS).to(device)
            output = torch.istft(output, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
                                   window=window,
                                   # onesided=True,
                                   return_complex=False,
                                   normalized=True,
                                   center=True)
            
            end_time = time.time()
            
            result_rtf_full.append((signal.shape[-1] / SR) / (end_time - start_time))
            nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
            result_nisqa_full.append(nisqa_score)

            for j in range(0, signal.shape[-1], window_size):
                chunk = signal[..., j:j+window_size]
                
                if chunk.shape[-1] < window_size:
                    continue

                start_time = time.time()
                spec = torch.stft(
                    chunk,
                    n_fft=N_FFTS,
                    hop_length=HOP_LENGTH,
                    # onesided=True,
                    win_length=N_FFTS,
                    window=window,
                    return_complex=True,
                    normalized=True,
                    center=True
                )

                input_spec = torch.view_as_real(spec)

                output = model(input_spec)

                output = torch.view_as_complex(output.contiguous())
                
                window = vorbis_window(N_FFTS).to(device)
                output = torch.istft(output, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
                                    window=window,
                                    # onesided=True,
                                    return_complex=False,
                                    normalized=True,
                                    center=True)
                
                end_time = time.time()


                result_rtf_chunk.append((chunk.shape[-1] / SR) / (end_time - start_time))
                # print(output.shape)
                # nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)

                # result_nisqa_chunk.append(nisqa_score)
                

    # print(f"Mean nisqa for full audio: ", torch.stack(result_nisqa_full).mean(dim=0))
    print(f"Mean rtf for full audio: ", torch.tensor(result_rtf_full).mean(dim=0), 1 / torch.tensor(result_rtf_full).mean(dim=0))
    print("---" * 10)
    # print("Mean nisqa for \"stream\" audio: ", torch.stack(result_nisqa_chunk).mean(dim=0))
    print("Mean rtf for \"stream\" audio: ", torch.tensor(result_rtf_chunk).mean(dim=-1), 1 / torch.tensor(result_rtf_chunk).mean(dim=-1))

    return result_nisqa_full, result_rtf_full, result_nisqa_chunk, result_rtf_chunk

In [12]:
_ = check_stream_inference(model, test_dataloader, device="cpu")

  0%|          | 0/824 [00:00<?, ?it/s]

100%|██████████| 824/824 [05:18<00:00,  2.59it/s]

Mean rtf for full audio:  tensor(30.574) tensor(0.033)
------------------------------
Mean rtf for "stream" audio:  tensor(18.180) tensor(0.055)


In [13]:
from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality
from torch_stoi import NegSTOILoss

srmr = SpeechReverberationModulationEnergyRatio(fs=16_000, norm=False)
pesq = PerceptualEvaluationSpeechQuality(fs=16_000, mode="wb").to("cuda")
stoi = NegSTOILoss(SR, use_vad=False, do_resample=False).to("cuda")
dnsmos = DeepNoiseSuppressionMeanOpinionScore(16_000, False, device=DEVICE)

In [14]:
from torchaudio.transforms import Resample
from thop import profile

from scipy.io.wavfile import write


def get_metrics(model, loader, device="cpu"):
    model.eval()
    
    model = model.to(device)
    
    nisqa_scores = []
    pesq_scores = []
    stoi_scores = []
    srmr_scores = []
    dnsmos_scores = []
    macs_list = []
    with torch.no_grad():
        for ind, (signal, target) in tqdm(enumerate(loader)):
            signal = signal.to(device)
            target = target.to(device)
            window = torch.hann_window(N_FFTS).pow(0.5).to(device) # vorbis_window(N_FFTS).to(device)
    
            spec = torch.stft(
                signal,
                n_fft=N_FFTS,
                hop_length=HOP_LENGTH,
                # onesided=True,
                win_length=N_FFTS,
                window=window,
                return_complex=True,
                normalized=True,
                center=True
            )

            input_spec = torch.view_as_real(spec)

            output = model(input_spec)
            macs_list.append(0)
            output_spec = torch.view_as_complex(output.contiguous())

            window = torch.hann_window(N_FFTS).pow(0.5).to(device) # vorbis_window(N_FFTS).to(device)
            output = torch.istft(output_spec, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
                                   window=window,
                                   # onesided=True,
                                   return_complex=False,
                                   normalized=True,
                                   center=True)
            
            # write(f'gtcrn_in/intput_{ind}.wav', SR, signal.cpu().detach().numpy()[0])
            # write(f'gtcrn_out/output_{ind}.wav', SR, output.cpu().detach().numpy()[0])
            
            # output = output / (output.abs().max() / signal.abs().max())
            
            min_l = min(output.shape[-1], signal.shape[-1])
            nisqa_score, _, _ = process(output.detach().cpu(), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
            stoi_score = stoi(output[..., :min_l], target[..., :min_l])
            
            resampler = Resample(SR, 16_000)
            output = resampler(output.cpu()).cuda()
            target = resampler(target.cpu()).cuda()
            min_l = min(output.shape[-1], target.shape[-1])

            srmr_score = srmr(output.detach().cpu())
            dnsmos_score = dnsmos(output.detach())

            try:
                pesq_score = pesq(output[..., :min_l], target[..., :min_l])
            except Exception as e:
                # print(min_l)
                # out_wave_ = output.reshape(-1)
                # target_ = target.reshape(-1)
                # write('exception_out.wav', SR, out_wave_.cpu().detach().numpy())
                # write('exception_in.wav', SR, target_.cpu().detach().numpy())
                continue

            nisqa_scores.append(nisqa_score[0])
            srmr_scores.append(srmr_score)
            stoi_scores.append(stoi_score.cpu())
            pesq_scores.append(pesq_score.cpu())
            dnsmos_scores.append(dnsmos_score.cpu())

    result = {"nisqa": nisqa_scores, "stoi": stoi_scores, "srmr": srmr_scores, "pesq": pesq_scores, "dnsmos": dnsmos_scores, "macs": macs_list}
        
    return result

In [15]:
metrics = get_metrics(model, test_dataloader, device="cuda")

824it [13:02,  1.05it/s]


In [16]:
print("NISQA:", torch.vstack(metrics["nisqa"]).mean(dim=0))
print("PESQ:", torch.vstack(metrics["pesq"]).mean(dim=0))
print("SRMR:", torch.vstack(metrics["srmr"]).mean(dim=0))
print("STOI:", -torch.vstack(metrics["stoi"]).mean(dim=0))
print("DNSMOS:", torch.vstack(metrics["dnsmos"]).mean(dim=0))
print("MACs:", sum(metrics["macs"]) / len(metrics["macs"]))

NISQA: tensor([2.041, 2.961, 2.545, 2.919, 3.030])
PESQ: tensor([2.103])
SRMR: tensor([8.223])
STOI: tensor([0.863])
DNSMOS: tensor([3.397, 3.027, 3.793, 2.691], dtype=torch.float64)
MACs: 0.0


In [17]:
input_sig, gt, gt_noise, gt_rir = dataset[0]

In [40]:
from thop import profile

window = vorbis_window(N_FFTS)

input_spec = torch.stft(
            input_sig[..., :SR],
            n_fft=N_FFTS,
            hop_length=HOP_LENGTH,
            # onesided=True,
            win_length=N_FFTS,
            window=window,
            return_complex=True,
            normalized=True,
            center=True
        )

input_spec = input_spec.to("cpu")
print(torch.view_as_real(input_spec).shape)

abs_spectrum = input_spec.abs()
input_spec_ = torch.permute(torch.view_as_real(input_spec), dims=(0, 2, 3, 1))
# print(input_spec_.shape)
batch, frames, channels, frequency = input_spec_.shape
abs_spectrum = torch.permute(abs_spectrum, dims=(0, 2, 1))
abs_spectrum = torch.reshape(abs_spectrum, shape=(batch, frames, 1, frequency))
# h0 = [[torch.zeros(configs.dual_path_extension["parameters"]["num_layers"], batch * configs.num_bands_out, configs.dual_path_extension["parameters"]["inter_hidden_size"], device=input_spec.device) for _ in range(8)] for _ in range(configs.dual_path_extension["num_modules"])]

# output, hid_out = fspen(input_spec_, abs_spectrum, h0)
# print(input_spec_.shape)

input_spec_ = torch.ones(1, 1, 257, 1, 2)

model.eval()

start = time.time()
macs, params = profile(model.cpu(), inputs=(input_spec_))
end = time.time()

torch.Size([1, 257, 63, 2])
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register count_prelu() for <class 'torch.nn.modules.activation.PReLU'>.
[INFO] Register count_gru() for <class 'torch.nn.modules.rnn.GRU'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.ConvTranspose2d'>.


In [41]:
print("MACs: ", macs)
print("Params: ", params)

MACs:  8113560.0
Params:  48245.0


In [42]:
from thop import profile

window = vorbis_window(N_FFTS)

input_spec = torch.stft(
            input_sig[..., :SR],
            n_fft=N_FFTS,
            hop_length=HOP_LENGTH,
            # onesided=True,
            win_length=N_FFTS,
            window=window,
            return_complex=True,
            normalized=True,
            center=True
        )

input_spec = input_spec.to("cpu")
print(torch.view_as_real(input_spec).shape)

abs_spectrum = input_spec.abs()
input_spec_ = torch.permute(torch.view_as_real(input_spec), dims=(0, 2, 3, 1))
# print(input_spec_.shape)
batch, frames, channels, frequency = input_spec_.shape
abs_spectrum = torch.permute(abs_spectrum, dims=(0, 2, 1))
abs_spectrum = torch.reshape(abs_spectrum, shape=(batch, frames, 1, frequency))
# h0 = [[torch.zeros(configs.dual_path_extension["parameters"]["num_layers"], batch * configs.num_bands_out, configs.dual_path_extension["parameters"]["inter_hidden_size"], device=input_spec.device) for _ in range(8)] for _ in range(configs.dual_path_extension["num_modules"])]

# output, hid_out = fspen(input_spec_, abs_spectrum, h0)
# print(input_spec_.shape)

input_spec_ = torch.ones(1, 1, 257, 2, 2)

model.eval()

start = time.time()
macs, params = profile(model.cpu(), inputs=(input_spec_))
end = time.time()

torch.Size([1, 257, 63, 2])
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register count_prelu() for <class 'torch.nn.modules.activation.PReLU'>.
[INFO] Register count_gru() for <class 'torch.nn.modules.rnn.GRU'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.ConvTranspose2d'>.


In [43]:
print("MACs: ", macs)
print("Params: ", params)

MACs:  16227120.0
Params:  48245.0
